In [2]:
# ==========================================
# 1️⃣ Import Libraries
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# ==========================================
# 2️⃣ Load Data
# ==========================================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

test_ids = test["Id"]

# ==========================================
# 3️⃣ Separate Target
# ==========================================
y = np.log1p(train["SalePrice"])
X = train.drop(["SalePrice"], axis=1)

# ==========================================
# 4️⃣ Combine Train + Test
# ==========================================
combined = pd.concat([X, test], axis=0)

# ==========================================
# 5️⃣ Handle Missing Values Smartly
# ==========================================
# Numerical
num_cols = combined.select_dtypes(include=["int64", "float64"]).columns
combined[num_cols] = combined[num_cols].fillna(combined[num_cols].median())

# Categorical
cat_cols = combined.select_dtypes(include=["object"]).columns
combined[cat_cols] = combined[cat_cols].fillna("None")

# ==========================================
# 6️⃣ One-Hot Encoding
# ==========================================
combined = pd.get_dummies(combined)

# ==========================================
# 7️⃣ Split Back
# ==========================================
X = combined[:len(train)]
test = combined[len(train):]

# ==========================================
# 8️⃣ Initialize Better Model
# ==========================================
model = GradientBoostingRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    max_features='sqrt',
    min_samples_leaf=15,
    min_samples_split=10,
    random_state=42
)

# ==========================================
# 9️⃣ Cross Validation (More Reliable)
# ==========================================
cv_scores = np.sqrt(-cross_val_score(
    model,
    X,
    y,
    scoring="neg_mean_squared_error",
    cv=5
))

print("Cross Validation RMSE:", cv_scores.mean())

# ==========================================
# 🔟 Train on Full Data
# ==========================================
model.fit(X, y)

# ==========================================
# 1️⃣1️⃣ Predict
# ==========================================
final_preds = model.predict(test)
final_preds = np.expm1(final_preds)

# ==========================================
# 1️⃣2️⃣ Submission
# ==========================================
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": final_preds
})

submission.to_csv("submission_improved.csv", index=False)

print("Improved submission file created 🚀")


Cross Validation RMSE: 0.12227169016659996
Improved submission file created 🚀
